## keyring import (api key 설정)
- (1) 미리 터미널에 입력해둔다.
- (2) python 코드 내에서 사용한다.

```bash
## bash 쉘 에서 다음 내용을 입력
## 형식 keyring set {{서비스명}} {{계정명}}

## e.g.
keyring set gemini-api-key---alpha300uk alpha300uk  
Password for 'alpha300uk' in 'gemini-api-key---alpha300uk':
```

In [4]:
import keyring
service_name = "gemini-api-key---alpha300uk"
username = "alpha300uk"
api_token = keyring.get_password(service_name, username)

## 의존성 설치 (혹시 설치 안했을 경우를 위해 추가한 섹션)

In [11]:
! pip install langgraph langchain langchain_google_genai langchain_community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typi

## llm, ratelimiter 선언

In [5]:
import os
os.environ['GOOGLE_API_KEY'] = api_token

from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI

# Gemini API는 분당 10개 요청으로 제한
# 즉, 초당 약 0.167개 요청 (10/60)
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)

# rate limiter를 LLM에 적용
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    rate_limiter=rate_limiter,
    # temperature
    # max_tokens

    thinking_budget = 500  # 추론(Reasoning) 토큰 길이 제한
)

1. `os.environ`: 환경 변수에 Google API 키를 설정합니다.
2. `InMemoryRateLimiter`: Gemini 무료 티어의 제한(분당 10회)을 준수하기 위해 초당 약 0.167회로 요청 속도를 제한하는 객체를 생성합니다.
3. `ChatGoogleGenerativeAI`: `gemini-2.5-flash` 모델을 초기화하며, 앞서 만든 `rate_limiter`를 적용하여 API 호출 안정성을 확보합니다. `thinking_budget`은 모델의 추론 프로세스에 할당할 토큰 한도를 설정합니다.

## e.g. 구조화된 출력 (1)
- 자료구조 `Objective` 생성 및 정의
- `llm.with_structured_output(Objective)`을 활용해 `Objective` 에 대해 구조화된 출력 생성

In [6]:
from pydantic import BaseModel, Field

# 프롬프트 자동 생성을 위한 요소 저장
class Objective(BaseModel):
    instruction: str = Field(description='프롬프트의 지시 사항을 명확히 재구성')
    output_format: str = Field(description='출력 포맷에 대한 설명')
    examples: str = Field(description='예시 출력(1개)')
    notes: str = Field(description='작업 과정에서 중요한 내용을 4개의 개조식 문장으로 구성')

    @property #
    def as_str(self) -> str:
        return '\n\n'.join([f'## {key}\n {value}' for key, value in self])


In [7]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate([
    ('system', '아래의 작업을 보다 자세하게 요청하는 시스템 프롬프트를 구성하고자 합니다. 주어진 포맷에 적절하게 작성하세요.'),
    ('user', '{instruction}')

])

chain = prompt | llm.with_structured_output(Objective)

result = chain.invoke("집에서 쉽게 구할 수 있는 재료로 재미있는 장난감 만들기")

result

Objective(instruction='사용자가 제시한 주제에 따라, 집에서 쉽게 구할 수 있는 재료만을 활용하여 만들 수 있는 재미있고 창의적인 장난감 만들기 아이디어를 제공하세요. 장난감의 이름, 필요한 재료 목록, 단계별 제작 방법, 그리고 안전 수칙을 명확하고 상세하게 설명해야 합니다.', output_format="출력은 JSON 객체 형식이어야 합니다. 객체는 'toy_name'(장난감 이름), 'materials'(필요한 재료 목록), 'instructions'(단계별 제작 방법), 'safety_tips'(안전 수칙)의 4가지 키를 포함해야 합니다. 'materials'와 'instructions'는 각 항목이 리스트 형태의 문자열로 구성되어야 합니다. 'safety_tips'는 3개 이상의 개조식 문장으로 구성된 문자열입니다.", examples='{"toy_name": "휴지심 망원경", "materials": ["휴지심 2개", "색종이 또는 포장지", "가위", "풀 또는 테이프", "끈 (선택 사항)"], "instructions": ["1. 휴지심 한쪽 끝을 약 1cm 정도 잘라 4등분으로 가위집을 냅니다.", "2. 다른 휴지심의 한쪽 끝을 1번과 동일하게 가위집을 냅니다.", "3. 두 휴지심의 가위집 낸 부분을 서로 교차시켜 끼워 고정합니다. (이때, 풀이나 테이프를 사용하여 더욱 단단히 고정할 수 있습니다.)", "4. 완성된 망원경 몸통에 색종이나 포장지를 붙여 예쁘게 꾸며줍니다.", "5. (선택 사항) 양쪽에 구멍을 뚫어 끈을 연결하면 목에 걸고 다닐 수 있는 망원경이 됩니다."], "safety_tips": "- 가위 사용 시 보호자의 감독 하에 주의하여 사용하세요.- 풀 또는 테이프 사용 후에는 손을 깨끗이 씻으세요.- 작은 부품이 떨어지지 않도록 튼튼하게 만드세요.- 만든 장난감을 입에 넣지 않도록 주의하세요."}', notes='- 제시된 주제에 충실하게 장난감 아이디어를 구체화해야 합니다.- 모든 재료는 가정에서

출력 예시 (출력이 나타나는 데까지 11초 가량 걸림)
```plain
Objective(instruction='사용자가 제시한 주제에 따라, 집에서 쉽게 구할 수 있는 재료만을 활용하여 만들 수 있는 재미있고 창의적인 장난감 만들기 아이디어를 제공하세요. 장난감의 이름, 필요한 재료 목록, 단계별 제작 방법, 그리고 안전 수칙을 명확하고 상세하게 설명해야 합니다.', output_format="출력은 JSON 객체 형식이어야 합니다. 객체는 'toy_name'(장난감 이름), 'materials'(필요한 재료 목록), 'instructions'(단계별 제작 방법), 'safety_tips'(안전 수칙)의 4가지 키를 포함해야 합니다. 'materials'와 'instructions'는 각 항목이 리스트 형태의 문자열로 구성되어야 합니다. 'safety_tips'는 3개 이상의 개조식 문장으로 구성된 문자열입니다.", examples='{"toy_name": "휴지심 망원경", "materials": ["휴지심 2개", "색종이 또는 포장지", "가위", "풀 또는 테이프", "끈 (선택 사항)"], "instructions": ["1. 휴지심 한쪽 끝을 약 1cm 정도 잘라 4등분으로 가위집을 냅니다.", "2. 다른 휴지심의 한쪽 끝을 1번과 동일하게 가위집을 냅니다.", "3. 두 휴지심의 가위집 낸 부분을 서로 교차시켜 끼워 고정합니다. (이때, 풀이나 테이프를 사용하여 더욱 단단히 고정할 수 있습니다.)", "4. 완성된 망원경 몸통에 색종이나 포장지를 붙여 예쁘게 꾸며줍니다.", "5. (선택 사항) 양쪽에 구멍을 뚫어 끈을 연결하면 목에 걸고 다닐 수 있는 망원경이 됩니다."], "safety_tips": "- 가위 사용 시 보호자의 감독 하에 주의하여 사용하세요.- 풀 또는 테이프 사용 후에는 손을 깨끗이 씻으세요.- 작은 부품이 떨어지지 않도록 튼튼하게 만드세요.- 만든 장난감을 입에 넣지 않도록 주의하세요."}', notes='- 제시된 주제에 충실하게 장난감 아이디어를 구체화해야 합니다.- 모든 재료는 가정에서 쉽게 찾을 수 있는 것들로 한정해야 합니다.- 제작 방법은 초등학생도 이해할 수 있도록 간단하고 명확한 단계로 구성해야 합니다.- 장난감의 안전한 사용을 위한 실질적인 안전 수칙을 반드시 포함해야 합니다.')
```

깔끔하게 출력

In [8]:
print(result.as_str)

## instruction
 사용자가 제시한 주제에 따라, 집에서 쉽게 구할 수 있는 재료만을 활용하여 만들 수 있는 재미있고 창의적인 장난감 만들기 아이디어를 제공하세요. 장난감의 이름, 필요한 재료 목록, 단계별 제작 방법, 그리고 안전 수칙을 명확하고 상세하게 설명해야 합니다.

## output_format
 출력은 JSON 객체 형식이어야 합니다. 객체는 'toy_name'(장난감 이름), 'materials'(필요한 재료 목록), 'instructions'(단계별 제작 방법), 'safety_tips'(안전 수칙)의 4가지 키를 포함해야 합니다. 'materials'와 'instructions'는 각 항목이 리스트 형태의 문자열로 구성되어야 합니다. 'safety_tips'는 3개 이상의 개조식 문장으로 구성된 문자열입니다.

## examples
 {"toy_name": "휴지심 망원경", "materials": ["휴지심 2개", "색종이 또는 포장지", "가위", "풀 또는 테이프", "끈 (선택 사항)"], "instructions": ["1. 휴지심 한쪽 끝을 약 1cm 정도 잘라 4등분으로 가위집을 냅니다.", "2. 다른 휴지심의 한쪽 끝을 1번과 동일하게 가위집을 냅니다.", "3. 두 휴지심의 가위집 낸 부분을 서로 교차시켜 끼워 고정합니다. (이때, 풀이나 테이프를 사용하여 더욱 단단히 고정할 수 있습니다.)", "4. 완성된 망원경 몸통에 색종이나 포장지를 붙여 예쁘게 꾸며줍니다.", "5. (선택 사항) 양쪽에 구멍을 뚫어 끈을 연결하면 목에 걸고 다닐 수 있는 망원경이 됩니다."], "safety_tips": "- 가위 사용 시 보호자의 감독 하에 주의하여 사용하세요.- 풀 또는 테이프 사용 후에는 손을 깨끗이 씻으세요.- 작은 부품이 떨어지지 않도록 튼튼하게 만드세요.- 만든 장난감을 입에 넣지 않도록 주의하세요."}

## notes
 - 제시된 주제에 충실하게 장난감 아이디어를 구체화해야 합니다.- 모든 재료는 가정에서 

출력 예시
```plain
## instruction
 사용자가 제시한 주제에 따라, 집에서 쉽게 구할 수 있는 재료만을 활용하여 만들 수 있는 재미있고 창의적인 장난감 만들기 아이디어를 제공하세요. 장난감의 이름, 필요한 재료 목록, 단계별 제작 방법, 그리고 안전 수칙을 명확하고 상세하게 설명해야 합니다.

## output_format
 출력은 JSON 객체 형식이어야 합니다. 객체는 'toy_name'(장난감 이름), 'materials'(필요한 재료 목록), 'instructions'(단계별 제작 방법), 'safety_tips'(안전 수칙)의 4가지 키를 포함해야 합니다. 'materials'와 'instructions'는 각 항목이 리스트 형태의 문자열로 구성되어야 합니다. 'safety_tips'는 3개 이상의 개조식 문장으로 구성된 문자열입니다.

## examples
 {"toy_name": "휴지심 망원경", "materials": ["휴지심 2개", "색종이 또는 포장지", "가위", "풀 또는 테이프", "끈 (선택 사항)"], "instructions": ["1. 휴지심 한쪽 끝을 약 1cm 정도 잘라 4등분으로 가위집을 냅니다.", "2. 다른 휴지심의 한쪽 끝을 1번과 동일하게 가위집을 냅니다.", "3. 두 휴지심의 가위집 낸 부분을 서로 교차시켜 끼워 고정합니다. (이때, 풀이나 테이프를 사용하여 더욱 단단히 고정할 수 있습니다.)", "4. 완성된 망원경 몸통에 색종이나 포장지를 붙여 예쁘게 꾸며줍니다.", "5. (선택 사항) 양쪽에 구멍을 뚫어 끈을 연결하면 목에 걸고 다닐 수 있는 망원경이 됩니다."], "safety_tips": "- 가위 사용 시 보호자의 감독 하에 주의하여 사용하세요.- 풀 또는 테이프 사용 후에는 손을 깨끗이 씻으세요.- 작은 부품이 떨어지지 않도록 튼튼하게 만드세요.- 만든 장난감을 입에 넣지 않도록 주의하세요."}

## notes
 - 제시된 주제에 충실하게 장난감 아이디어를 구체화해야 합니다.- 모든 재료는 가정에서 쉽게 찾을 수 있는 것들로 한정해야 합니다.- 제작 방법은 초등학생도 이해할 수 있도록 간단하고 명확한 단계로 구성해야 합니다.- 장난감의 안전한 사용을 위한 실질적인 안전 수칙을 반드시 포함해야 합니다.
```

## e.g. (2)

In [17]:
### 예전 코드 --- (start)
import keyring
import os

from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel, Field

service_name = "gemini-api-key---alpha300uk"
username = "alpha300uk"

# 키링에서 토큰 조회
api_token = keyring.get_password(service_name, username)

# api_token 등록
os.environ['GOOGLE_API_KEY'] = api_token

# Gemini API는 분당 10개 요청으로 제한
# 즉, 초당 약 0.167개 요청 (10/60)
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)

# rate limiter를 LLM에 적용
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    rate_limiter=rate_limiter,
    # temperature
    # max_tokens

    thinking_budget = 500  # 추론(Reasoning) 토큰 길이 제한
)

# 프롬프트 자동 생성을 위한 요소 저장
class Objective(BaseModel):
    instruction: str = Field(description='프롬프트의 지시 사항을 명확히 재구성')
    output_format: str = Field(description='출력 포맷에 대한 설명')
    examples: str = Field(description='예시 출력(1개)')
    notes: str = Field(description='작업 과정에서 중요한 내용을 4개의 개조식 문장으로 구성')

    @property #
    def as_str(self) -> str:
        return '\n\n'.join([f'## {key}\n {value}' for key, value in self])


from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate([
    ('system', '아래의 작업을 보다 자세하게 요청하는 시스템 프롬프트를 구성하고자 합니다. 주어진 포맷에 적절하게 작성하세요.'),
    ('user', '{instruction}')

])
### 예전 코드 --- (end)

from typing import TypedDict

class State(TypedDict):
    instruction : str
    prompt_materials : Objective # Objective Class를 하나의 값에 저장
    full_prompt : str
    result : str


def get_prompt_materials(State):
    prompt = ChatPromptTemplate([
        ('system', '아래의 작업을 보다 자세하게 세분화하고자 합니다. 주어진 포맷에 적절하게 작성하세요.'),
        ('user', '{instruction}')

    ])

    chain = prompt | llm.with_structured_output(Objective)

    result = chain.invoke({'instruction':State['instruction']})
    return {'prompt_materials' : result}


from langchain_core.output_parsers import StrOutputParser

def generate_prompt(State):
    prompt = ChatPromptTemplate([
        ('system', '''당신은 체계적이고 정확한 프롬프트 엔지니어입니다. 아래의 포인트를 바탕으로, LLM에 입력할  시스템 프롬프트를 작성하세요.
{points}'''),
        ('user', '{instruction}')
    ])

    chain = prompt | llm | StrOutputParser()

    result = chain.invoke({'instruction': State['instruction'], 'points': State['prompt_materials'].as_str})
    return {'full_prompt' : result}

def generate(State):
    return {'result' : llm.invoke(State['full_prompt']).content}


from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END


# 그래프 구성
builder = StateGraph(State)

builder.add_node("get_prompt_materials", get_prompt_materials)
builder.add_node("generate_prompt", generate_prompt)
builder.add_node("generate", generate)

builder.add_edge(START, "get_prompt_materials")
builder.add_edge("get_prompt_materials", "generate_prompt")
builder.add_edge("generate_prompt", "generate")

builder.add_edge("generate", END)
graph = builder.compile()


import pprint

# Streaming 참고
# https://langchain-ai.github.io/langgraph/concepts/streaming/#streaming-graph-outputs-stream-and-astream

for data in graph.stream({'instruction': '''영화 '마이너리티 리포트'와 AI 윤리의 연관성에 대한 리포트 쓰기'''},
                         stream_mode='values'):
    pprint.pprint(data)
    print('----')

{'instruction': "영화 '마이너리티 리포트'와 AI 윤리의 연관성에 대한 리포트 쓰기"}
----
{'instruction': "영화 '마이너리티 리포트'와 AI 윤리의 연관성에 대한 리포트 쓰기",
 'prompt_materials': Objective(instruction="영화 '마이너리티 리포트'의 주요 내용과 '프리크라임' 시스템을 분석하고, 이 시스템이 제기하는 윤리적 딜레마를 현대 AI 윤리의 핵심 쟁점들(예: 예측 편향, 자유 의지 침해, 책임 소재, 알고리즘의 투명성)과 구체적으로 연결하여 리포트를 작성하십시오.", output_format='서론, 본론(영화 내용 요약 및 AI 윤리 쟁점 분석), 결론으로 구성된 리포트 형식으로 작성합니다.', examples="서론: 영화 '마이너리티 리포트'는 미래 사회의 범죄 예방 시스템 '프리크라임'을 통해 AI 기술이 가져올 수 있는 윤리적 딜레마를 심도 있게 탐구합니다. 본론: 프리크라임 시스템은 예지 능력을 가진 '프리콥스'를 활용하여 범죄가 발생하기 전에 범인을 체포함으로써 사회의 안전을 극대화합니다. 그러나 이는 아직 발생하지 않은 '미래의 범죄'로 인해 개인의 자유 의지를 침해하고, 무고한 사람을 처벌할 수 있다는 심각한 윤리적 문제를 내포합니다. 특히, AI가 미래를 예측하고 그 예측에 기반하여 인간의 삶을 결정하는 것이 정당한지에 대한 질문은 현대 AI 윤리 논의에서 '예측 편향', '책임 소재', '알고리즘의 투명성' 등과 깊이 연결됩니다. 결론: '마이너리티 리포트'는 AI 기반 예측 시스템이 가져올 수 있는 사회적, 윤리적 파장을 미리 보여주며, 기술 발전과 함께 윤리적 고려가 필수적임을 강조합니다.", notes="영화 '마이너리티 리포트'의 핵심 줄거리와 '프리크라임' 시스템의 작동 방식을 간략하게 요약해야 합니다.AI 윤리 쟁점은 영화 속 상황과 직접적으로 연결하여 구체적인 사례를 들어 설명해야 합니다.자유 의지, 결정론, 그리고 예측 기술의 한계에 대한 철학적 논

## 2. Message 포맷의 State 사용하기
State의 저장값으로 Message를 바로 사용하기도 합니다.   
이 경우, Context에 메시지를 계속 추가하거나, 별도의 로직을 만들어 메시지 정보를 전달합니다.
`typing`의 `Annotated`로 공간을 지정한 후, 뒷부분에 결합 로직을 포함합니다.   
이를 리듀서(Reducer)라고 부르는데, 메시지의 경우 아래와 같이 포함하면 됩니다.

In [ ]:
### 예전 코드 --- (start)
import keyring
import os

from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel, Field

service_name = "gemini-api-key---alpha300uk"
username = "alpha300uk"

# 키링에서 토큰 조회
api_token = keyring.get_password(service_name, username)

# api_token 등록
os.environ['GOOGLE_API_KEY'] = api_token

# Gemini API는 분당 10개 요청으로 제한
# 즉, 초당 약 0.167개 요청 (10/60)
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)

# rate limiter를 LLM에 적용
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    rate_limiter=rate_limiter,
    # temperature
    # max_tokens

    thinking_budget = 500  # 추론(Reasoning) 토큰 길이 제한
)

# 프롬프트 자동 생성을 위한 요소 저장
class Objective(BaseModel):
    instruction: str = Field(description='프롬프트의 지시 사항을 명확히 재구성')
    output_format: str = Field(description='출력 포맷에 대한 설명')
    examples: str = Field(description='예시 출력(1개)')
    notes: str = Field(description='작업 과정에서 중요한 내용을 4개의 개조식 문장으로 구성')

    @property #
    def as_str(self) -> str:
        return '\n\n'.join([f'## {key}\n {value}' for key, value in self])


from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate([
    ('system', '아래의 작업을 보다 자세하게 요청하는 시스템 프롬프트를 구성하고자 합니다. 주어진 포맷에 적절하게 작성하세요.'),
    ('user', '{instruction}')

])
### 예전 코드 --- (end)


### 이번 코드 --- (start)

import pprint

# Streaming 참고
# https://langchain-ai.github.io/langgraph/concepts/streaming/#streaming-graph-outputs-stream-and-astream
def pprint_graph(graph):
    for data in graph.stream({'instruction': '''영화 '마이너리티 리포트'와 AI 윤리의 연관성에 대한 리포트 쓰기'''},
                            stream_mode='values'):
        pprint.pprint(data)
        print('----')

from typing import Annotated
from typing import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    context: Annotated[list, add_messages]

builder = StateGraph(State)
builder.add_node('talk',talk)
builder.add_edge(START, 'talk')
builder.add_edge('talk', END)

graph = builder.compile()
pprint_graph(graph)

messages = [
    SystemMessage(content='시스템 메시지 1'),
    HumanMessage(content='유저 메시지 1'),
    AIMessage(content='AI 메시지 1'),
    HumanMessage(content='유저 메시지 2'),
]

response = graph.invoke({'context': messages})
# print(response)


### 메시지 삭제 
from langchain_core.messages import RemoveMessage
def delete_message(State):
    # 첫번째,두번째 메시지 삭제
    messages = State['context']
    return {"context": [RemoveMessage(id = messages[i].id) for i in range(1,3)]}


builder = StateGraph(State)

builder.add_node('talk',talk)
builder.add_node('delete_message',delete_message)

builder.add_edge(START, 'talk')
builder.add_edge('talk', 'delete_message')
builder.add_edge('delete_message', END)
graph = builder.compile()
# print(graph)
pprint_graph(graph)

# print(messages)

# 유저 메시지 1, AI 메시지 1 삭제
graph.invoke({'context': messages})
### 이번 코드 --- (end)